In [ ]:
# Import necessary libraries
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from keras.models import Sequential
import tensorflow as tf
from keras.layers import Conv2D, MaxPooling2D, Dropout, Flatten, Dense, Conv2DTranspose, Cropping2D, UpSampling2D, Reshape

from tensorflow.keras.preprocessing.image import ImageDataGenerator

from tensorflow.keras.applications.vgg16 import VGG16
from tensorflow.keras.applications.xception import Xception
from tensorflow.keras.applications.inception_v3 import InceptionV3
from tensorflow.keras.models import Model



In [ ]:
latent_dim = 512

In [ ]:
target_shape = (8, 10, 64)

In [ ]:


# Define the VAE architecture
class VAE(tf.keras.Model):
    def __init__(self, latent_dim):
        super(VAE, self).__init__()
        self.latent_dim = latent_dim
        self.encoder = keras.Sequential([
            layers.InputLayer(input_shape=(400, 320, 3)),
            Conv2D(128, (3, 3), activation='relu', padding='same'),
            MaxPooling2D((2, 2), padding='same'),
            Conv2D(64, (3, 3), activation='relu', padding='same'),
            MaxPooling2D((2, 2), padding='same'),
            Conv2D(64, (3, 3), activation='relu', padding='same'),
            MaxPooling2D((2, 2), padding='same'),
            Conv2D(64, (3, 3), activation='relu', padding='same'),
            MaxPooling2D((2, 2), padding='same'),
            Conv2D(32, (3, 3), activation='relu', padding='same'),
            MaxPooling2D((2, 2), padding='same'),
            Conv2D(32, (3, 3), activation='relu', padding='same'),
            MaxPooling2D((2, 2), padding='same'),
            Flatten(),
            Dense(latent_dim + latent_dim),
            ])


        self.decoder = keras.Sequential([
            layers.InputLayer(input_shape=(latent_dim,)),
            Dense(target_shape[0]*target_shape[1]*target_shape[2], activation='relu'),
            Reshape(target_shape),
            Conv2DTranspose(256, (3, 3), strides=(2, 2), activation='relu', padding='same'),
            Conv2DTranspose(128, (3, 3), strides=(2, 2), activation='relu', padding='same'),
            Conv2DTranspose(64, (3, 3), strides=(2, 2), activation='relu', padding='same'),
            Conv2DTranspose(64, (3, 3), strides=(2, 2), activation='relu', padding='same'),
            Conv2DTranspose(32, (3, 3), strides=(2, 2), activation='relu', padding='same'),
            Conv2DTranspose(16, (3, 3), strides=(2, 2), activation='relu', padding='same'),
            Conv2D(3, (3, 3), activation='sigmoid', padding='same'),
            Cropping2D(cropping=((56,56), (160,160))),

        ])
        


    def sample(self, eps=None):
        if eps is None:
            eps = tf.random.normal(shape=(100, self.latent_dim))
        return self.decode(eps, apply_sigmoid=True)

    def encode(self, x):
        mean, logvar = tf.split(self.encoder(x), num_or_size_splits=2, axis=1)
        return mean, logvar

    def reparameterize(self, mean, logvar):
        eps = tf.random.normal(shape=mean.shape)
        return eps * tf.exp(logvar * 0.5) + mean

    def decode(self, z, apply_sigmoid=False):
        logits = self.decoder(z)
        if apply_sigmoid:
            probs = tf.sigmoid(logits)
            return probs
        return logits

In [ ]:
# Custom loss function for VAE
@tf.function
def compute_loss(model, x):
    mean, logvar = model.encode(x)
    z = model.reparameterize(mean, logvar)
    x_logit = model.decode(z)

    cross_ent = tf.nn.sigmoid_cross_entropy_with_logits(logits=x_logit, labels=x)
    logpx_z = -tf.reduce_sum(cross_ent, axis=[1, 2, 3])
    logpz = tf.reduce_sum(tf.square(z), axis=1)
    logqz_x = -tf.reduce_sum(0.5 * (logvar + tf.square(mean) - logvar), axis=1)
    return -tf.reduce_mean(logpx_z + logpz - logqz_x)

# Training step function
@tf.function
def train_step(model, x, optimizer):
    with tf.GradientTape() as tape:
        loss = compute_loss(model, x)
    gradients = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(gradients, model.trainable_variables))
    return loss



In [ ]:
# Training loop
def train_vae(model, dataset, optimizer, epochs):
    for epoch in range(epochs):
        for train_x in range(len(dataset)):
            loss = train_step(model, dataset[train_x], optimizer)
        print('Epoch: {}, Loss: {:.4f}'.format(epoch + 1, loss)) 

In [ ]:
input_shape = (400,320, 3)
datagen = ImageDataGenerator(rescale=1./255)
    
path_train = "/path/to/private-data/Datagen_Left/Inlet (5 marking)/train"
path_val = "/path/to/private-data/Datagen_Left/Inlet (5 marking)/validation"
    
batch_size = 32
target_size = (input_shape[0],input_shape[1])
train_generator = datagen.flow_from_directory(path_train, target_size=target_size,  batch_size=batch_size, class_mode=None, shuffle=True, color_mode='rgb')

validation_generator = datagen.flow_from_directory(path_val, target_size=target_size , batch_size=batch_size, class_mode=None, shuffle=True, color_mode='rgb')

In [ ]:
model = VAE(latent_dim=latent_dim)

In [ ]:
len(train_generator)

In [ ]:

optimizer = tf.keras.optimizers.Adam(0.0001)
train_vae(model,train_generator, optimizer, 10)